# 04. Tree-based and bagging ensembles: Decision Tree, Random Forest, Extra Trees
**Run on:** Laptop A · **Approx. runtime:** ~8-12 min on a 4-core laptop


**Protocol (same for every model):**
- Each row is one *(GPS point, candidate arc)* pair. The label is 1 if the arc is the point's reference arc.
- **5-fold GroupKFold by trip** on the 78 usable training trips. Each fold fits on a 300k-row sample of points (KNN uses 100k) and is scored on all candidates of the held-out trips.
- The model is then **refit on the training split** and scored on the **10 student-test trips**.
- **Metrics:**
  - macro F1 / precision / recall: candidate level, threshold 0.5
  - log loss
  - **Top-1 accuracy**: the share of GPS points whose highest-scoring candidate is the reference arc
  - continuity
- The **HMM/Viterbi** decoder (parameters in `artifacts/hmm_params.json`) is then run on this model's probabilities to compare *classifier only* with *classifier + HMM*.


**Before running:** the dataset must be in `data/train/<id>/` and `data/student_test/<id>/`, and `pip install -r requirements.txt` done. Run all cells top to bottom (Kernel → Restart & Run All). Every number and plot below is computed when you run it.

In [ ]:
import os, sys, warnings
# run from notebooks/ or from the repo root
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
warnings.filterwarnings("ignore")
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160)
from pathsnap import nb
from pathsnap.models import make_model
from pathsnap.config import CFG
SEED = CFG["seed"]  # 42: every model and data split uses this seed
print("repo root:", ROOT)

In [ ]:
tr, tr_pts, te, te_pts = nb.load_data()

## Decision Tree


**What it does:** a flowchart of yes/no splits on features (e.g. *dist_minus_min ≤ 0?*, *heading_diff ≤ 40°?*) ending in leaves with a probability.
- **Why it suits PathSnap:** captures feature interactions and thresholds naturally; no scaling needed.
- **Limitation:** one tree is high-variance, so it is regularised here.
- **Hyperparameters:** max_depth = 12, min_samples_leaf = 50.
- **Objective:** Gini impurity reduction at each split.

In [ ]:
make_dtree = lambda: make_model("dtree", SEED)
print(make_dtree())
cv_dtree = nb.run_cv(make_dtree, tr, tr_pts, max_rows=300000)
cv_dtree.round(4)

In [ ]:
model_dtree, train_dtree, test_dtree, p_dtree = nb.fit_and_evaluate(make_dtree, tr, tr_pts, te, te_pts, max_rows=300000)
print(f"student-test continuity (classifier only): {test_dtree['continuity']:.4f} | inference {test_dtree['infer_ms_per_point']:.4f} ms/point")
nb.summary_table("Decision Tree", cv_dtree, train_dtree, test_dtree)

In [ ]:
imp_dtree = nb.plot_importance(model_dtree, te, "Decision Tree")

In [ ]:
hmm_dtree = nb.hmm_compare(model_dtree, te, "Decision Tree")
nb.plot_hmm(hmm_dtree, "Decision Tree: classifier only vs classifier + HMM (student-test trips)")
nb.save_results("dtree", cv_dtree, train_dtree, test_dtree,
                {"hmm_mean": hmm_dtree.drop(columns=["trip", "model"]).mean().to_dict()})

## Random Forest


**What it does:** 200 decision trees, each trained on a bootstrap sample with random feature subsets (sqrt of 14) at each split; probabilities are averaged.
- **Why:** averaging de-correlated trees reduces the variance of a single tree.
- **Limitation:** large model files (about 18 MB) and slower than boosting at similar accuracy.
- **Hyperparameters:** n_estimators = 200, max_depth = 16, min_samples_leaf = 20.
- **Objective:** Gini impurity per tree.

In [ ]:
make_rf = lambda: make_model("rf", SEED)
print(make_rf())
cv_rf = nb.run_cv(make_rf, tr, tr_pts, max_rows=300000)
cv_rf.round(4)

In [ ]:
model_rf, train_rf, test_rf, p_rf = nb.fit_and_evaluate(make_rf, tr, tr_pts, te, te_pts, max_rows=300000)
print(f"student-test continuity (classifier only): {test_rf['continuity']:.4f} | inference {test_rf['infer_ms_per_point']:.4f} ms/point")
nb.summary_table("Random Forest", cv_rf, train_rf, test_rf)

In [ ]:
imp_rf = nb.plot_importance(model_rf, te, "Random Forest")

In [ ]:
hmm_rf = nb.hmm_compare(model_rf, te, "Random Forest")
nb.plot_hmm(hmm_rf, "Random Forest: classifier only vs classifier + HMM (student-test trips)")
nb.save_results("rf", cv_rf, train_rf, test_rf,
                {"hmm_mean": hmm_rf.drop(columns=["trip", "model"]).mean().to_dict()})

## Extra Trees


**What it does:** like Random Forest, but split thresholds are drawn **at random** and the best random split is kept, which adds even more randomisation.
- **Why tried:** often as accurate as RF and faster to train.
- **Limitation:** random thresholds are less precise on the key distance and heading features; here it is the weakest tree ensemble.
- **Hyperparameters:** n_estimators = 200, max_depth = 18, min_samples_leaf = 20.
- **Objective:** Gini impurity.

In [ ]:
make_extratrees = lambda: make_model("extratrees", SEED)
print(make_extratrees())
cv_extratrees = nb.run_cv(make_extratrees, tr, tr_pts, max_rows=300000)
cv_extratrees.round(4)

In [ ]:
model_extratrees, train_extratrees, test_extratrees, p_extratrees = nb.fit_and_evaluate(make_extratrees, tr, tr_pts, te, te_pts, max_rows=300000)
print(f"student-test continuity (classifier only): {test_extratrees['continuity']:.4f} | inference {test_extratrees['infer_ms_per_point']:.4f} ms/point")
nb.summary_table("Extra Trees", cv_extratrees, train_extratrees, test_extratrees)

In [ ]:
imp_extratrees = nb.plot_importance(model_extratrees, te, "Extra Trees")

In [ ]:
hmm_extratrees = nb.hmm_compare(model_extratrees, te, "Extra Trees")
nb.plot_hmm(hmm_extratrees, "Extra Trees: classifier only vs classifier + HMM (student-test trips)")
nb.save_results("extratrees", cv_extratrees, train_extratrees, test_extratrees,
                {"hmm_mean": hmm_extratrees.drop(columns=["trip", "model"]).mean().to_dict()})

## Comparison of the models in this notebook

In [ ]:
nb.plot_cv_folds({"Decision Tree": cv_dtree, "Random Forest": cv_rf, "Extra Trees": cv_extratrees})
nb.plot_overfit({"Decision Tree": (train_dtree, cv_dtree, test_dtree), "Random Forest": (train_rf, cv_rf, test_rf), "Extra Trees": (train_extratrees, cv_extratrees, test_extratrees)})
nb.plot_confusion({"Decision Tree": test_dtree["confusion"], "Random Forest": test_rf["confusion"], "Extra Trees": test_extratrees["confusion"]})
nb.plot_calibration(te.label.to_numpy(), {"Decision Tree": p_dtree, "Random Forest": p_rf, "Extra Trees": p_extratrees})

In [ ]:
rows = []
for name, cv, t, h in [("Decision Tree", cv_dtree, test_dtree, hmm_dtree), ("Random Forest", cv_rf, test_rf, hmm_rf), ("Extra Trees", cv_extratrees, test_extratrees, hmm_extratrees)]:
    rows.append({"model": name, "CV macro F1": cv.macro_f1.mean(), "CV std": cv.macro_f1.std(),
                 "test Top-1": t["top1"], "test macro F1": t["macro_f1"], "test log loss": t["log_loss"],
                 "test continuity": t["continuity"], "Top-1 + HMM (trip mean)": h.hmm_top1.mean(),
                 "route F1 + HMM (trip mean)": h.hmm_route_f1.mean()})
pd.DataFrame(rows).set_index("model").round(4)